# **Baseline: Neural Surrogate Model (MLP) + Kalman Filter**

### **Mount Google Drive**

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

### **Setup Project Structure**

In [ ]:
from pathlib import Path

PROJECT_DIR = Path('/content/drive/MyDrive/ML_for_WDNs/project_4/wdn_baseline')
WORK_DIR = Path('/content/baseline_neural_surrogate')

print('Project directory:', PROJECT_DIR)
print('Working directory:', WORK_DIR)

### **Install Project Dependencies**

In [ ]:
%pip install --upgrade --no-cache-dir \
    epyt-flow==0.17.2 \
    epyt-control \
    water-benchmark-hub \
    seaborn

### **Experiment Setup**

In [ ]:
from dataclasses import dataclass
from pathlib import Path
from typing import Sequence
import os
import sys
import zipfile
import shutil
import random
import subprocess
import numpy as np
import matplotlib.pyplot as plt

@dataclass(frozen=True)
class Dataset:
    """Datasets for baseline experiment."""
    net_name: str
    file_prefix: str
    sensors: Sequence[int]


NET1 = Dataset(
    net_name='Net1',
    file_prefix='net1',
    sensors=list(range(2, 10)),
)

HANOI = Dataset(
    net_name='Hanoi',
    file_prefix='hanoi',
    sensors=list(range(2, 26)),
)


@dataclass
class BaselinePaths:
    """set paths for repo, data, trained models and experiment results."""
    project_dir: Path
    work_dir: Path = Path('/content/baseline_neural_surrogate')
    repo_name: str = 'NeuralSurrogateKalmanChlorineEstimation'
    repo_url: str = 'https://github.com/andreArtelt/NeuralSurrogateKalmanChlorineEstimation.git'

    @property
    def repo_dir(self) -> Path:
        return self.work_dir / self.repo_name

    @property
    def data_dir(self) -> Path:
        return self.repo_dir / 'data'

    @property
    def model_output(self) -> Path:
        return self.project_dir / 'trained_surrogates'

    @property
    def result_output(self) -> Path:
        return self.project_dir / 'baseline_results'

    @property
    def generated_data_zip(self) -> Path:
        return self.project_dir / 'generated_data.zip'

    def create_dirs(self) -> None:
        self.work_dir.mkdir(parents=True, exist_ok=True)
        self.model_output.mkdir(parents=True, exist_ok=True)
        self.result_output.mkdir(parents=True, exist_ok=True)


class RepoSetup:
    """Clone or update baseline repo, load data and setup."""
    def __init__(self, paths: BaselinePaths):
        self.paths = paths

    def get_repo(self) -> None:
        self.paths.work_dir.mkdir(parents=True, exist_ok=True)

        if self.paths.repo_dir.exists():
            print("Repo already exists. Updating ...")
            subprocess.run(
                ["git", "-C", str(self.paths.repo_dir), "pull"],
                check=True,
            )
        else:
            print("Repo does not yet exist. Cloning repo ...")
            subprocess.run(
                ["git", "clone", self.paths.repo_url, str(self.paths.repo_dir)],
                check=True,
            )

    def load_data(self, overwrite: bool = False) -> None:
        if not self.paths.generated_data_zip.exists():
            raise FileNotFoundError(f"No data found: {self.paths.generated_data_zip}")

        if overwrite and self.paths.data_dir.exists():
            shutil.rmtree(self.paths.data_dir)

        self.paths.data_dir.mkdir(parents=True, exist_ok=True)

        print(f"Load data into {self.paths.data_dir}")
        with zipfile.ZipFile(self.paths.generated_data_zip, "r") as z:
            z.extractall(self.paths.data_dir)

    def set_work_dir(self) -> None:
        repo_path = str(self.paths.repo_dir)

        if repo_path not in sys.path:
            sys.path.insert(0, repo_path)

        os.chdir(self.paths.repo_dir)
        print("Current working directory:", Path.cwd())

    def setup(self, load_data: bool = True, overwrite_data: bool = False) -> None:
        self.paths.create_dirs()
        self.get_repo()

        if load_data:
            self.load_data(overwrite=overwrite_data)

        self.set_work_dir()


@dataclass
class ExperimentResult:
    dataset: Dataset
    sensors: Sequence[int]
    mean_scores: np.ndarray
    std_scores: np.ndarray
    n_iters: int
    seed: int
    result_file: Path


class BaselineExperiment:
    """Runs surrogate training and EKF experiments."""

    def __init__(self, paths: BaselinePaths):
        self.paths = paths

    # File paths
    def training_scada_file(self, dataset: Dataset) -> str:
        return os.path.join('data', f'{dataset.file_prefix}_randDemand=True_training.epytflow_scada_data')

    def training_actions_file(self, dataset: Dataset) -> str:
        return os.path.join('data', f'{dataset.file_prefix}_randDemand=True_training.npz')

    def test_scada_file(self, dataset: Dataset) -> str:
        return os.path.join('data', f'{dataset.file_prefix}_randDemand=False_test.epytflow_scada_data')

    def test_actions_file(self, dataset: Dataset) -> str:
        return os.path.join('data', f'{dataset.file_prefix}_randDemand=False_test.npz')

    def local_surrogate_file(self, dataset: Dataset) -> str:
        return os.path.join('data', f'{dataset.file_prefix}_randDemand=True_surrogate.pt')

    def drive_surrogate_file(self, dataset: Dataset) -> Path:
        return self.paths.model_output / f'{dataset.file_prefix}_randDemand=True_surrogate.pt'

    def local_surrogate_scaler_file(self, dataset: Dataset) -> Path:
        return Path(f"{self.local_surrogate_file(dataset)}.pickle")

    def drive_surrogate_scaler_file(self, dataset: Dataset) -> Path:
        return Path(f"{self.drive_surrogate_file(dataset)}.pickle")

    def result_file(self, dataset: Dataset, n_iters: int, seed: int) -> Path:
        return self.paths.result_output / f'baseline_{dataset.file_prefix}_n_iters={n_iters}_seed={seed}.npz'

    def train_surrogate(
        self,
        dataset: Dataset,
    ) -> Path:
        from fit_surrogates import fit_surrogate

        local_model = Path(
            self.local_surrogate_file(dataset)
        )

        local_scaler = (
            self.local_surrogate_scaler_file(dataset)
        )

        print(
            f"Training surrogate for "
            f"{dataset.net_name}..."
        )

        fit_surrogate(
            net_desc=dataset.net_name,
            scada_file_in=(
                self.training_scada_file(dataset)
            ),
            control_actions_file_in=(
                self.training_actions_file(dataset)
            ),
            file_out=str(local_model),
        )

        if not local_model.exists():
            raise FileNotFoundError(
                f"Training did not create model: "
                f"{local_model}"
            )

        if not local_scaler.exists():
            raise FileNotFoundError(
                f"Training did not create scaler: "
                f"{local_scaler}"
            )

        drive_model = (
            self.drive_surrogate_file(dataset)
        )

        drive_scaler = (
            self.drive_surrogate_scaler_file(dataset)
        )

        shutil.copy2(
            local_model,
            drive_model,
        )

        shutil.copy2(
            local_scaler,
            drive_scaler,
        )

        print(
            f"Saved model to Drive: "
            f"{drive_model}"
        )

        print(
            f"Saved scaler to Drive: "
            f"{drive_scaler}"
        )

        return drive_model

    def restore_surrogate(
        self,
        dataset: Dataset,
    ) -> Path:
        drive_model = (
            self.drive_surrogate_file(dataset)
        )

        drive_scaler = (
            self.drive_surrogate_scaler_file(dataset)
        )

        local_model = Path(
            self.local_surrogate_file(dataset)
        )

        local_scaler = (
            self.local_surrogate_scaler_file(dataset)
        )

        missing_files = [
            path
            for path in (
                drive_model,
                drive_scaler,
            )
            if not path.exists()
        ]

        if missing_files:
            missing_text = "\n".join(
                str(path)
                for path in missing_files
            )

            raise FileNotFoundError(
                "The saved surrogate is incomplete. "
                "Missing:\n"
                f"{missing_text}\n\n"
                "Retrain the surrogate once so both "
                "the model and scaler are saved."
            )

        local_model.parent.mkdir(
            parents=True,
            exist_ok=True,
        )

        shutil.copy2(
            drive_model,
            local_model,
        )

        shutil.copy2(
            drive_scaler,
            local_scaler,
        )

        print(
            f"Restored {dataset.net_name} model to: "
            f"{local_model}"
        )

        print(
            f"Restored {dataset.net_name} scaler to: "
            f"{local_scaler}"
        )

        return local_model

    def get_surrogate(
        self,
        dataset: Dataset,
        force_retrain: bool = False,
    ) -> Path:
        drive_model = (
            self.drive_surrogate_file(dataset)
        )

        drive_scaler = (
            self.drive_surrogate_scaler_file(dataset)
        )

        complete_saved_surrogate = (
            drive_model.exists()
            and drive_scaler.exists()
        )

        if (
            complete_saved_surrogate
            and not force_retrain
        ):
            return self.restore_surrogate(
                dataset
            )

        return self.train_surrogate(
            dataset
        )

    # Experiment
    def run_experiments(self, dataset: Dataset, n_iters: int, seed: int = 0) -> ExperimentResult:
        from run_exp_state_estimation import run_exp
        if not Path(self.local_surrogate_file(dataset)).exists():
            self.restore_surrogate(dataset)

        random.seed(seed)
        np.random.seed(seed)

        print(f'Running {dataset.net_name}: {len(dataset.sensors)} sensors × {n_iters} sensor placements')

        mean_scores, std_scores = run_exp(
            n_sensors_range=dataset.sensors,
            n_iters=n_iters,
            net_desc=dataset.net_name,
            scada_file_in=self.test_scada_file(dataset),
            control_actions_file_in=self.test_actions_file(dataset),
            state_transition_model_file_in=self.local_surrogate_file(dataset),
        )

        output_file = (self.paths.result_output/ (f"{dataset.file_prefix}_ekf_"f"n_iters={n_iters}_seed={seed}.npz"))

        for n, mean, std in zip(dataset.sensors, mean_scores, std_scores):
            print(f'{n} sensors/type: {mean:.4f} ± {std:.4f}')

        return ExperimentResult(
            dataset=dataset,
            sensors=dataset.sensors,
            mean_scores=mean_scores,
            std_scores=std_scores,
            n_iters=n_iters,
            seed=seed,
            result_file=output_file
        )

class BaselinePlot:
    """Plot baseline experiment results."""
    def __init__(self, paths: BaselinePaths):
        self.paths = paths

    def plot_file(
        self,
        result: ExperimentResult,
        kalman_type: str,
    ) -> Path:
        return (
            self.paths.result_output
            / f'baseline_{result.dataset.file_prefix}_{kalman_type}_n_iters={result.n_iters}_seed={result.seed}.png'
        )

    def plot_experiment_results(self, result: ExperimentResult, kalman_type: str) -> Path:
        plot_file = self.plot_file(result, kalman_type)

        plt.figure(figsize=(7, 4))
        plt.errorbar(
            result.sensors,
            result.mean_scores,
            yerr=result.std_scores,
            marker='o',
            capsize=4,
        )
        plt.xlabel('Number of sensors per type')
        plt.ylabel('Chlorine estimation error')
        plt.title(f'Baseline {kalman_type} + Neural Surrogate: 'f'{result.dataset.net_name}')
        plt.grid(True)
        plt.savefig(plot_file, dpi=200, bbox_inches='tight')
        plt.show()

        print(f'Saved plot to: {plot_file}')
        return plot_file

### **Train or Load Models**

In [ ]:
paths = BaselinePaths(
    project_dir=PROJECT_DIR,
    work_dir=WORK_DIR,
)

repo_setup = RepoSetup(paths)

repo_setup.setup(
    load_data=False,
)

experiment = BaselineExperiment(paths)
plotter = BaselinePlot(paths)

In [ ]:
import os
import subprocess
import sys

CREATE_DATA_SCRIPT = PROJECT_DIR / "create_data.py"

if not CREATE_DATA_SCRIPT.exists():
    raise FileNotFoundError(
        f"Could not find data-generation script: {CREATE_DATA_SCRIPT}"
    )

os.chdir(paths.repo_dir)

print("Working directory:", os.getcwd())
print("Running script:", CREATE_DATA_SCRIPT)

subprocess.run(
    [sys.executable, str(CREATE_DATA_SCRIPT)],
    check=True,
)

print("Data generation finished.")

In [ ]:
import shutil
from pathlib import Path

archive_base = PROJECT_DIR / "generated_data"

archive_path = shutil.make_archive(
    base_name=str(archive_base),
    format="zip",
    root_dir=str(paths.data_dir),
)

print("Saved generated data to:", archive_path)

In [ ]:
# Net1
net1_model = experiment.get_surrogate(
    dataset=NET1,
    force_retrain=False,
)

print("Net1 model ready:", net1_model)

In [ ]:
# Hanoi
hanoi_model = experiment.get_surrogate(
    dataset=HANOI,
    force_retrain=True,
)

print("Hanoi model ready:", hanoi_model)

### **Run Experiments**

In [ ]:
from pathlib import Path
import numpy as np


def save_experiment_result(
    result: ExperimentResult,
    output_file: str | Path,
    filter_name: str,
) -> Path:
    """Save an ExperimentResult as a compressed NumPy file."""
    output_file = Path(output_file)

    output_file.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    np.savez_compressed(
        output_file,
        filter_name=filter_name,
        dataset=result.dataset.net_name,
        sensors=np.asarray(result.sensors),
        mean_scores=np.asarray(result.mean_scores),
        std_scores=np.asarray(result.std_scores),
        n_iters=result.n_iters,
        seed=result.seed,
    )

    return output_file


def load_experiment_result(
    input_file: str | Path,
    dataset: Dataset,
) -> ExperimentResult:
    """Load a saved result for plotting and comparison."""
    input_file = Path(input_file)

    with np.load(
        input_file,
        allow_pickle=False,
    ) as saved:
        return ExperimentResult(
            dataset=dataset,
            sensors=saved["sensors"].tolist(),
            mean_scores=saved["mean_scores"].copy(),
            std_scores=saved["std_scores"].copy(),
            n_iters=int(saved["n_iters"]),
            seed=int(saved["seed"]),
            result_file=input_file,
        )

In [ ]:
# run Net1 experiment
net1_result_ekf = experiment.run_experiments(
    dataset=NET1,
    n_iters=30, # SET TO 30 FOR REAL TEST!!!
    seed=42,
)

In [ ]:
net1_result_file = save_experiment_result(
    result=net1_result_ekf,
    output_file=(
        paths.result_output
        / "net1_ekf_n_iters=30_seed=42.npz"
    ),
    filter_name="EKF",
)

In [ ]:
# run Hanoi experiments
hanoi_result = experiment.run_experiments(
    dataset=HANOI,
    n_iters=30, # SET TO 30 FOR REAL TEST!!!
    seed=42,
)

In [ ]:
hanoi_result_file = save_experiment_result(
    result=hanoi_result,
    output_file=(
        paths.result_output
        / "hanoi_ekf_n_iters=30_seed=42.npz"
    ),
    filter_name="EKF",
)

### **Plot Results**

In [ ]:
# load net1 results
net1_result = load_experiment_result(input_file=(paths.result_output/ "net1_ekf_n_iters=30_seed=42.npz"),
    dataset=NET1,
)

# plotting Net1
plotter.plot_experiment_results(net1_result, kalman_type='EKF')

In [ ]:
# load hanoi
hanoi_result = load_experiment_result(input_file=(paths.result_output/ "hanoi_ekf_n_iters=30_seed=42.npz"),
    dataset=HANOI,
)

# plotting Hanoi
plotter.plot_experiment_results(hanoi_result, kalman_type="EKF")

### **Ensemble Kalman Filter**

In [ ]:
# imports
from dataclasses import dataclass

In [ ]:
@dataclass
class EnKFConfig:
    """Filter configuration."""
    ensemble_size: int = 50
    seed: int = 42

In [ ]:
class GaussianNoiseModel:
    """Set up different covariance matrices to model uncertainty."""

    def __init__(
        self,
        state_dim,
        obs_dim,
        initial_covariance=None,
        process_covariance=None,
        measurement_covariance=None,
        seed=42,
    ):
        """initialise different noise matrices."""
        self.state_dim = state_dim
        self.obs_dim = obs_dim
        self.seed = seed
        self.initial_covariance = (np.eye(state_dim) if initial_covariance is None else initial_covariance)
        self.process_covariance = (np.eye(state_dim)if process_covariance is None else process_covariance)
        self.measurement_covariance = (np.eye(obs_dim) if measurement_covariance is None else measurement_covariance)
        self.rng = np.random.default_rng(seed)

    def sample_initial_noise(self, count):
        """Sample from initial covariance matrix P."""
        return self.rng.multivariate_normal(
            mean=np.zeros(self.state_dim),
            cov=self.initial_covariance,
            size=count,
        )

    def sample_process_noise(self, count):
        """Sample from process covariance matrix Q."""
        return self.rng.multivariate_normal(
            mean=np.zeros(self.state_dim),
            cov=self.process_covariance,
            size=count,
        )

    def sample_measurement_noise(self, count):
        """Sample from measurement covariance matrix R."""
        return self.rng.multivariate_normal(
            mean=np.zeros(self.obs_dim),
            cov=self.measurement_covariance,
            size=count,
        )

    def reset(self):
        """Reset random number generator."""
        self.rng = np.random.default_rng(self.seed)

In [ ]:
class EnsembleState:
    """Store ensemble members and calculate ensemble statistics."""

    def __init__(
        self,
        initial_state,
        ensemble_size,
        noise_model,
    ):
        """initialise ensemble."""
        self.initial_state = np.asarray(initial_state, dtype=float).reshape(-1)
        self.ensemble_size = ensemble_size
        self.noise_model = noise_model
        self.reset_mean()

    @property
    def mean(self):
        """return ensemble mean (best filter estimate)."""
        return self.members.mean(axis=0)

    @property
    def covariance(self):
        """Capture uncertainty and correlations of variables."""
        return np.cov(self.members, rowvar=False, ddof=1)

    def reset_mean(self):
        """regenerate the ensemble so it is centred on the initial state."""
        noise = self.noise_model.sample_initial_noise(self.ensemble_size)
        noise -= noise.mean(axis=0, keepdims=True)
        self.members = (self.initial_state.reshape(1, -1) + noise)

    def replace_members(self, new_members):
        """add new ensemble members."""
        self.members = np.asarray(new_members, dtype=float)

    def add_changes(self, changes):
        """add changes to ensemble members."""
        self.members += np.asarray(changes, dtype=float)

    def shift_mean(self, desired_mean):
        """shift ensemble mean to desired value, preserving spread."""
        desired_mean = np.asarray(desired_mean, dtype=float).reshape(-1)
        self.members += (desired_mean - self.mean).reshape(1, -1)

    def set_values(self, indices, values):
        """
        Shift ensemble so mean matches observed flows.
        """
        indices = np.asarray(indices, dtype=int)
        values = np.asarray(values, dtype=float)
        shifts = (np.asarray(values)- self.mean[indices])
        self.members[:, indices] += shifts

In [ ]:
class TimeVaryingEnsembleKalmanFilter:
    """
    Set up time-varying EnKF.
    """

    def __init__(
        self,
        state_dim,
        obs_dim,
        init_state,
        get_state_transition_func,
        get_measurement_func,
        init_state_uncertainty_cov=None,
        measurement_uncertainty_cov=None,
        system_uncertainty_cov=None,
        config=None,
    ):
        """set up Ensemble Kalman Filter with parameters."""
        self.state_dim = state_dim
        self.obs_dim = obs_dim
        self.config = config or EnKFConfig()

        self._get_state_transition_func = (get_state_transition_func)

        self._get_measurement_func = (get_measurement_func)

        self._initial_state = np.asarray(
            init_state,
            dtype=float,
            )

        self._noise = GaussianNoiseModel(
            state_dim=state_dim,
            obs_dim=obs_dim,
            initial_covariance=init_state_uncertainty_cov,
            process_covariance=system_uncertainty_cov,
            measurement_covariance=measurement_uncertainty_cov,
            seed=self.config.seed,
            )

        self._state = EnsembleState(
            initial_state=self._initial_state,
            ensemble_size=self.config.ensemble_size,
            noise_model=self._noise,
        )

        self._time_step = 0
        self._x = self._state.mean.copy()

    @property
    def time_step(self):
        """Return the current internal filter time index."""
        return self._time_step

    @property
    def ensemble(self):
        """return ensemble members."""
        return self._state.members.copy()

    @property
    def mean(self):
        """return ensemble mean."""
        return self._state.mean.copy()

    @property
    def covariance(self):
        """return ensemble covariance."""
        return self._state.covariance.copy()

    def step(self, observation):
        """
        Perform one prediction and correction step.

        Returns:
            corrected state mean
            corrected state covariance
        """
        self._state.shift_mean(self._x)
        transition = self._get_state_transition_func(self._time_step)
        measurement = self._get_measurement_func(self._time_step)

        self._predict(transition)
        self._correct(observation, measurement)

        self._x = self._state.mean.copy()
        self._time_step += 1

        return (
            self._x.copy(),
            self._state.covariance.copy(),
        )

    def _predict(self, transition):
        """Propagate members through the surrogate model and add Q noise."""
        predicted_members = np.array([transition(member) for member in self._state.members])
        process_noise = (self._noise.sample_process_noise(self.config.ensemble_size))
        self._state.replace_members(predicted_members + process_noise)

    def _correct(self, observation, measurement):
        """Correct ensemble members using the sensor observations."""
        predicted_observations = np.array([
            measurement(member)
            for member in self._state.members
        ])
        state_anomalies = self._state.members - self._state.mean

        observation_anomalies = predicted_observations - predicted_observations.mean(axis=0)

        denominator = self.config.ensemble_size - 1

        cross_covariance = state_anomalies.T @ observation_anomalies / denominator

        observation_covariance = observation_anomalies.T @ observation_anomalies / denominator + self._noise.measurement_covariance

        kalman_gain = np.linalg.solve(observation_covariance.T, cross_covariance.T).T

        measurement_noise = self._noise.sample_measurement_noise(self.config.ensemble_size)

        perturbed_observations = np.asarray(observation) + measurement_noise

        innovations = perturbed_observations - predicted_observations

        corrections = innovations @ kalman_gain.T

        self._state.add_changes(corrections)

    def set_state_values(self, indices, values):
        """insert known state values (e.g. measured flows)."""
        self._state.set_values(indices=indices,values=values)
        self._x = self._state.mean.copy()

    def reset(self):
        """Reset ensemble, random seed and time step."""
        self._noise.reset()
        self._state.reset_mean()
        self._time_step = 0
        self._x = self._state.mean.copy()

### **Run Ensemble Kalman Filter**

In [ ]:
@dataclass(frozen=True)
class EnKFExperimentConfig:
    num_node_quality_sensors: int = 2
    num_link_sensors: int = 2
    ensemble_size: int = 50
    initial_variance: float = 0.01
    process_variance: float = 0.001
    measurement_variance: float = 0.01
    seed: int = 42

In [ ]:
@dataclass
class EnKFStateEstimationResult:
    dataset: Dataset
    chlorine_scores: np.ndarray
    chlorine_predictions: np.ndarray
    chlorine_true: np.ndarray
    chlorine_std: np.ndarray
    sensor_matrix: np.ndarray
    flow_indices: np.ndarray

    @property
    def num_steps(self):
        return len(self.chlorine_scores)

    @property
    def mean_chlorine_score(self):
        return float(
            np.mean(self.chlorine_scores)
        )

In [ ]:
class EnKFStateEstimator:
    """
    Run the custom EnKF with Scada data and the trained neural surrogate model.
    """
    def __init__(
        self,
        experiment,
        dataset,
        config,
    ):
        self.experiment = experiment
        self.dataset = dataset
        self.config = config

    def run(self, max_steps=None):
        """Run EnKF over given time sequence."""
        self._prepare_repository_data()
        enkf = self._create_filter()
        available_steps = (len(self.states_scaled) - 1)

        if max_steps is None:
            num_steps = available_steps
        else:
            num_steps = min(
                max_steps,
                available_steps,
            )

        scores = []
        predictions = []
        true_values = []
        uncertainties = []

        for target_index in range(1, num_steps + 1):
            true_state_scaled = (self.states_scaled[target_index])

            enkf.set_state_values(indices=self.flow_indices,
                                  values=true_state_scaled[self.flow_indices]
                                  )

            observation = (self.sensor_matrix @ true_state_scaled)

            enkf.step(observation)

            ensemble_physical = (self.convert_to_physical_units(enkf.ensemble))

            estimated_state = (ensemble_physical.mean(axis=0))

            estimated_std = (ensemble_physical.std(axis=0, ddof=1))

            true_state = self.convert_to_physical_units(true_state_scaled)[0]

            predicted_chlorine = (estimated_state[:self.num_chlorine_states])

            actual_chlorine = (true_state[:self.num_chlorine_states])

            predictions.append(predicted_chlorine)

            true_values.append(actual_chlorine)

            uncertainties.append(estimated_std[:self.num_chlorine_states])

            scores.append(np.median(np.abs(predicted_chlorine - actual_chlorine)))

        return EnKFStateEstimationResult(
            dataset=self.dataset,
            chlorine_scores=np.asarray(scores),
            chlorine_predictions=np.vstack(predictions),
            chlorine_true=np.vstack(true_values),
            chlorine_std=np.vstack(uncertainties),
            sensor_matrix=(self.sensor_matrix.copy()),
            flow_indices=(self.flow_indices.copy())
        )

    def _prepare_repository_data(self):
        """Load and prepare the repository data once."""
        from epyt_flow.simulation import ScadaData

        from run_exp_state_estimation import (
            create_random_sensor_placement,
            get_state_transition_model,
        )

        scada_file = Path(self.experiment.test_scada_file(self.dataset))
        control_file = Path(self.experiment.test_actions_file(self.dataset))
        model_file = Path(self.experiment.local_surrogate_file(self.dataset))
        scaler_file = Path(f"{model_file}.pickle")

        if (not model_file.exists() or not scaler_file.exists()):
            self.experiment.restore_surrogate(self.dataset)

        for path in (scada_file, control_file, model_file, scaler_file):
            if not path.exists():
                raise FileNotFoundError(path)

        scada = ScadaData.load_from_file(str(scada_file))
        flows = scada.get_data_flows()
        node_quality = (scada.get_data_nodes_quality())
        link_quality = (scada.get_data_links_quality())

        with np.load(control_file) as data:
            control_actions = data["control_actions"]

        states_physical = np.concatenate((node_quality[:-1], link_quality[:-1], flows[1:]), axis=1,)
        self.controls = control_actions[:len(states_physical)]
        self.num_nodes = node_quality.shape[1]
        self.num_links = link_quality.shape[1]
        self.num_chlorine_states = (self.num_nodes + self.num_links)
        self.state_dim = (states_physical.shape[1])
        self.model = get_state_transition_model(self.dataset.net_name, str(model_file))
        self.model.n_missing_flows = (self.num_chlorine_states)
        self.model._normalize_input_output = False

        state_and_control = np.concatenate((states_physical, self.controls), axis=1)

        self.states_scaled = self.model._scaler.transform(state_and_control)[:, :self.state_dim]

        (self.sensor_matrix, flow_indices) = create_random_sensor_placement(
            n_node_quality_sensors=(
                self.config
                .num_node_quality_sensors
            ),
            n_link_sensors=(
                self.config.num_link_sensors
            ),
            n_nodes=self.num_nodes,
            n_links=self.num_links,
            state_dim=self.state_dim,
        )

        self.flow_indices = np.asarray(flow_indices, dtype=int)
        self.observation_dim = (self.sensor_matrix.shape[0])

    def _create_filter(self):
        """Create EnKF for run."""
        def get_measurement_function(time_step):
            return lambda state: (
                self.sensor_matrix @ np.asarray(state).reshape(-1)
            )

        def get_transition_function(time_step):
            control = self.controls[time_step + 1].reshape(1, -1)

            def transition(state):
                return (self.model.predict(np.asarray(state).reshape(1,-1), control).flatten())

            return transition

        return TimeVaryingEnsembleKalmanFilter(
            state_dim=self.state_dim,
            obs_dim=self.observation_dim,
            init_state=self.states_scaled[0],
            get_state_transition_func=(get_transition_function),
            get_measurement_func=(get_measurement_function),
            init_state_uncertainty_cov=(self.config.initial_variance * np.eye(self.state_dim)),
            system_uncertainty_cov=(self.config.process_variance * np.eye(self.state_dim)),
            measurement_uncertainty_cov=(self.config.measurement_variance * np.eye(self.observation_dim)),
            config=EnKFConfig(ensemble_size=(self.config.ensemble_size), seed=self.config.seed))

    def convert_to_physical_units(
        self,
        scaled_states,
    ):
        """Convert scaled states back to physical units."""
        scaled_states = np.atleast_2d(scaled_states)
        control_padding = np.zeros((len(scaled_states), self.controls.shape[1]))
        padded_states = np.concatenate((scaled_states, control_padding), axis=1)

        return (
            self.model._scaler.inverse_transform(padded_states)[:, :self.state_dim]
        )

In [ ]:
def run_enkf_experiments(
    experiment,
    dataset,
    n_iters,
    seed=0,
    ensemble_size=50,
    initial_variance=0.01,
    process_variance=0.001,
    measurement_variance=0.01,
):
    """Run EnKF for the same sensor counts and placements as the EKF."""
    random.seed(seed)
    np.random.seed(seed)

    mean_scores = []
    std_scores = []

    print(
        f"Running {dataset.net_name} EnKF: "
        f"{len(dataset.sensors)} sensor counts × "
        f"{n_iters} sensor placements"
    )

    for n_sensors in dataset.sensors:
        scores = []

        for _ in range(n_iters):
            config = EnKFExperimentConfig(
                num_node_quality_sensors=n_sensors,
                num_link_sensors=n_sensors,
                ensemble_size=ensemble_size,
                initial_variance=initial_variance,
                process_variance=process_variance,
                measurement_variance=measurement_variance,
                seed=seed,
            )

            estimator = EnKFStateEstimator(
                experiment=experiment,
                dataset=dataset,
                config=config,
            )

            result = estimator.run(
                max_steps=None,
            )

            scores.append(
                result.chlorine_scores
            )

        mean_score = np.mean(scores)
        std_score = np.std(scores)

        mean_scores.append(mean_score)
        std_scores.append(std_score)

        print(
            f"{n_sensors} sensors/type: "
            f"{mean_score:.4f} ± {std_score:.4f}"
        )

    mean_scores = np.asarray(mean_scores)
    std_scores = np.asarray(std_scores)

    output_file = (
        paths.result_output
        / (
            f"baseline_{dataset.file_prefix}_enkf_"
            f"n_iters={n_iters}_seed={seed}.npz"
        )
    )

    experiment_result = ExperimentResult(
        dataset=dataset,
        sensors=dataset.sensors,
        mean_scores=mean_scores,
        std_scores=std_scores,
        n_iters=n_iters,
        seed=seed,
        result_file=output_file,
    )

    save_experiment_result(
        result=experiment_result,
        output_file=output_file,
        filter_name="EnKF",
    )

    print(f"Saved results to: {output_file}")

    return experiment_result


In [ ]:
net1_result_enkf = run_enkf_experiments(
    experiment=experiment,
    dataset=NET1,
    n_iters=30,
    seed=42,
    ensemble_size=50,
    initial_variance=0.01,
    process_variance=0.001,
    measurement_variance=0.01,
)

In [ ]:
plotter = BaselinePlot(paths)

plotter.plot_experiment_results(
    net1_result_enkf,
    kalman_type="EnKF",
)

In [ ]:
print(type(net1_result_enkf))
print(net1_result_enkf)

In [ ]:
print(
    "Steps:",
    net1_result_enkf.num_steps,
)

print(
    "Mean chlorine error:",
    net1_result_enkf.mean_chlorine_score,
)

print(
    "Prediction shape:",
    net1_result_enkf
    .chlorine_predictions
    .shape,
)

print(
    "True-value shape:",
    net1_result_enkf
    .chlorine_true
    .shape,
)

print(
    "Uncertainty shape:",
    net1_result_enkf
    .chlorine_std
    .shape,
)